# Tune `threshold.json` trên Kaggle (checkpoint từ `gs://meta-cxr-checkpoint`)

Môi trường: **Kaggle VM**.

Nguồn dữ liệu:
- **Ảnh + CheXpert CSV + split**: Kaggle dataset `phuong20052/mimic-cxr-jpg-lite`.
- **CSV đã processed (train/val/test)**: Kaggle dataset `phuong20052/mimic-cxr-p10-processed`.
- **Checkpoint**: GCS bucket `gs://meta-cxr-checkpoint` (cần Internet bật trong Kaggle settings).

Output:
- `/kaggle/working/threshold.json`
- `/kaggle/working/threshold_search_details.csv`

Logic tune giống `META_CXR_generate_thresholds_kaggle.ipynb`; chỉ thay phần lấy checkpoint.

## 1. Dependencies

In [ ]:
# Bật Internet trong Kaggle Settings → "Internet On"
!pip install -q \
  "numpy<2" "opencv-python<4.10" "omegaconf==2.3.0" \
  iopath timm pandas scikit-image accelerate sentencepiece protobuf \
  iterative-stratification einops fairscale pycocoevalcap webdataset decord \
  ftfy regex hi-ml-multimodal torchinfo google-cloud-storage

## 2. Định vị code META-CXR + Kaggle datasets

In [ ]:
import os
import shutil
import sys
from pathlib import Path

WORK_DIR  = Path('/kaggle/working')
INPUT_DIR = Path('/kaggle/input')

def is_meta_cxr_project(path: Path) -> bool:
    return (path / 'model' / 'lavis').exists() and (path / 'pretraining').exists()

candidates = [Path.cwd(), WORK_DIR / 'META-CXR']
if INPUT_DIR.exists():
    for root in INPUT_DIR.glob('*'):
        candidates.extend([root, root / 'META-CXR'])

source_project = next((p for p in candidates if is_meta_cxr_project(p)), None)
if source_project is None:
    raise FileNotFoundError('Không tìm thấy code META-CXR. Attach dataset/source chứa thư mục META-CXR.')

PROJECT_DIR = WORK_DIR / 'META-CXR'
if source_project.resolve() != PROJECT_DIR.resolve() and not is_meta_cxr_project(PROJECT_DIR):
    shutil.copytree(
        source_project, PROJECT_DIR, dirs_exist_ok=True,
        ignore=shutil.ignore_patterns('.git', 'wandb', '__pycache__', '*.pyc', 'output', 'outputs', 'checkpoints'),
    )

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'model'))

def first_existing(paths: list[str]) -> Path:
    for item in paths:
        p = Path(item)
        if p.exists():
            return p
    raise FileNotFoundError('Không tìm thấy path: ' + ', '.join(paths))

IMAGE_ROOT = first_existing([
    '/kaggle/input/mimic-cxr-jpg-lite',
    '/kaggle/input/datasets/mimic-cxr-jpg-lite',
])
PROCESSED_ROOT = first_existing([
    '/kaggle/input/mimic-cxr-p10-processed',
    '/kaggle/input/datasets/mimic-cxr-p10-processed',
])

print('PROJECT_DIR    =', PROJECT_DIR)
print('IMAGE_ROOT     =', IMAGE_ROOT)
print('PROCESSED_ROOT =', PROCESSED_ROOT)

## 3. Tải checkpoint từ `gs://meta-cxr-checkpoint`

Auth bằng **Kaggle Secrets** (Add-ons → Secrets):
- Tên secret: `GCP_SERVICE_ACCOUNT_JSON`
- Value: toàn bộ nội dung Service Account JSON (paste nguyên `{...}`).

Notebook đọc secret → tạo `storage.Client` qua `from_service_account_info`. Không upload key file.

In [ ]:
import json as _json
from google.cloud import storage
from google.oauth2 import service_account
from kaggle_secrets import UserSecretsClient

GCS_BUCKET = 'meta-cxr-checkpoint'
CHECKPOINT_DIR = WORK_DIR / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

sa_json_str = UserSecretsClient().get_secret('GCP_SERVICE_ACCOUNT_JSON')
sa_info = _json.loads(sa_json_str)
credentials = service_account.Credentials.from_service_account_info(sa_info)
client = storage.Client(project=sa_info.get('project_id'), credentials=credentials)
bucket = client.bucket(GCS_BUCKET)

def download_blob(blob_name: str, dest: Path) -> Path:
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists():
        return dest
    bucket.blob(blob_name).download_to_filename(str(dest))
    return dest

blobs = list(client.list_blobs(GCS_BUCKET))
print(f'Tìm thấy {len(blobs)} blob trong gs://{GCS_BUCKET}:')
for b in blobs[:30]:
    print(f'  {b.name}  ({(b.size or 0)/1e6:.1f} MB)')

In [ ]:
# Chọn checkpoint cần tải. Đổi RUN_NAME để match prefix tên file trong bucket.
RUN_NAME = '07_all_three'

candidates = [
    b for b in blobs
    if RUN_NAME in b.name and b.name.endswith(('checkpoint_best.pth', 'checkpoint_last.pth'))
]
if not candidates:
    raise FileNotFoundError(f'Không có checkpoint_best/last cho {RUN_NAME} trong gs://{GCS_BUCKET}. Liệt kê ở cell trên.')

# Ưu tiên best, fallback last
best = [b for b in candidates if b.name.endswith('checkpoint_best.pth')]
chosen = sorted(best or candidates, key=lambda b: len(b.name))[0]

local_ckpt = CHECKPOINT_DIR / Path(chosen.name).name
print(f'Downloading gs://{GCS_BUCKET}/{chosen.name} → {local_ckpt}')
download_blob(chosen.name, local_ckpt)
print(f'OK ({local_ckpt.stat().st_size / 1e6:.1f} MB)')

## 4. Ghi `env_config.yaml`

In [ ]:
(PROJECT_DIR / 'configs').mkdir(exist_ok=True)
(PROJECT_DIR / 'configs' / 'env_config.yaml').write_text(f'''paths:
  data_root: "{IMAGE_ROOT}"
  mimic_cxr_jpg_root: "{IMAGE_ROOT}"
  split_csv: "{IMAGE_ROOT}/mimic-cxr-2.0.0-split.csv"
  reports_csv: "/kaggle/working/mimic_cxr_cleaned.csv"
  chexpert_csv: "{IMAGE_ROOT}/mimic-cxr-2.0.0-chexpert.csv"
  metadata_csv: "{IMAGE_ROOT}/mimic-cxr-2.0.0-metadata.csv"
  processed_dir: "{PROCESSED_ROOT}"
  processed_train_csv: "{PROCESSED_ROOT}/train.csv"
  processed_val_csv: "{PROCESSED_ROOT}/val.csv"
  processed_test_csv: "{PROCESSED_ROOT}/test.csv"
  output_dir: "/kaggle/working/output"
  checkpoint_dir: "{CHECKPOINT_DIR}"
wandb:
  entity: ""
  project: "meta-cxr-encoder-comparison"
java:
  home: "/usr/lib/jvm/java-11-openjdk-amd64"
  path: "/usr/lib/jvm/java-11-openjdk-amd64/bin:"
''', encoding='utf-8')
print('wrote env_config.yaml')

## 5. Build model + chạy forward trên val split

In [ ]:
import gc
import json
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

import model.lavis.tasks as tasks
from model.lavis.common.config import Config
from model.lavis.common.registry import registry
from model.lavis.common.optims import LinearWarmupCosineLRScheduler, LinearWarmupStepLRScheduler  # noqa: F401
from model.lavis.datasets.builders import *  # noqa: F401,F403
from model.lavis.models import *              # noqa: F401,F403
from model.lavis.processors import *          # noqa: F401,F403
from model.lavis.tasks import *               # noqa: F401,F403
from model.lavis.data.ReportDataset import MIMIC_CXR_Dataset
from local_config import VIS_ROOT

registry.mapping['paths']['cache_root'] = '.'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

CHEXPERT_COLS = [
    'No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity',
    'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis',
    'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices',
]
CLASS_MAP = {'negative': 0, 'positive': 1, 'uncertain': 2}

SPLIT = 'val'                # KHÔNG tune trên test
THRESHOLD_GRID = [round(x / 100, 2) for x in range(1, 100)]
EVAL_BATCH_SIZE = 4
NUM_WORKERS = 2

print('DEVICE =', DEVICE)

In [ ]:
def build_cfg(run_name: str) -> Config:
    cfg_path = PROJECT_DIR / 'pretraining' / 'configs' / 'encoder_comparison' / f'{run_name}.yaml'
    return Config(SimpleNamespace(cfg_path=str(cfg_path), options=None))

def build_model_for_run(run_name: str, checkpoint_path: Path):
    cfg = build_cfg(run_name)
    task = tasks.setup_task(cfg)
    model = task.build_model(cfg)
    ckpt = torch.load(checkpoint_path, map_location='cpu')
    state_dict = ckpt['model'] if isinstance(ckpt, dict) and 'model' in ckpt else ckpt
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    print(f'{run_name}: missing={len(missing)} unexpected={len(unexpected)}')
    return cfg, model.to(DEVICE).eval()

def make_loader(cfg, split: str) -> DataLoader:
    ds = MIMIC_CXR_Dataset(
        vis_processor=None, text_processor=None,
        vis_root=VIS_ROOT, split=split, cfg=cfg, truncate=None,
    )
    return DataLoader(ds, batch_size=EVAL_BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=(DEVICE == 'cuda'))

@torch.no_grad()
def predict_logits(model, batch) -> torch.Tensor:
    image = batch['image'].to(DEVICE, non_blocking=True)
    text = batch['text_output']
    cnn_p, vit_p, swin_p, _ = model._encode_image_streams(image, apply_aug=False)
    tok = model.tokenizer(text, padding='max_length', truncation=True,
                          max_length=model.max_txt_len, return_tensors='pt').to(DEVICE)
    text_out = model.Qformer.bert(tok.input_ids, attention_mask=tok.attention_mask, return_dict=True)
    logits, *_ = model.mhcac(cnn_patches=cnn_p, vit_patches=vit_p, swin_patches=swin_p,
                              text_embeddings=text_out.last_hidden_state, labels=None)
    return logits

In [ ]:
cfg, model = build_model_for_run(RUN_NAME, local_ckpt)
loader = make_loader(cfg, SPLIT)

all_probs, all_labels = [], []
for batch in tqdm(loader, desc=f'{RUN_NAME}:{SPLIT}'):
    logits = predict_logits(model, batch)
    all_probs.append(torch.softmax(logits, dim=-1).cpu().numpy())
    all_labels.append(batch['classification_labels'].cpu().numpy())

probs = np.concatenate(all_probs, axis=0)
labels = np.concatenate(all_labels, axis=0)
print('probs', probs.shape, 'labels', labels.shape)

del model, loader; gc.collect()
if DEVICE == 'cuda':
    torch.cuda.empty_cache()

## 6. Grid-search threshold + lưu kết quả

In [ ]:
def best_t(y_bin: np.ndarray, scores: np.ndarray, grid: list[float]) -> tuple[float, float]:
    best, best_f1 = 0.5, -1.0
    for t in grid:
        f1 = f1_score(y_bin, (scores >= t).astype(int), zero_division=0)
        if f1 > best_f1:
            best, best_f1 = float(t), float(f1)
    return best, best_f1

thresholds: dict[str, dict[str, float]] = {}
detail_rows: list[dict] = []
for ti, abn in enumerate(CHEXPERT_COLS):
    thresholds[abn] = {}
    y_task = labels[:, ti]
    for cls_name, cls_idx in CLASS_MAP.items():
        y_bin = (y_task == cls_idx).astype(int)
        if int(y_bin.sum()) == 0:
            continue
        t, f1 = best_t(y_bin, probs[:, ti, cls_idx], THRESHOLD_GRID)
        thresholds[abn][cls_name] = t
        detail_rows.append({
            'abnormality': abn, 'class': cls_name,
            'threshold': t, 'val_f1': f1,
            'positive_count': int(y_bin.sum()), 'n': len(y_bin),
        })

out_json = WORK_DIR / 'threshold.json'
out_csv  = WORK_DIR / 'threshold_search_details.csv'
out_json.write_text(json.dumps(thresholds, indent=4), encoding='utf-8')
pd.DataFrame(detail_rows).to_csv(out_csv, index=False)
print('saved', out_json)
print('saved', out_csv)
print(json.dumps(thresholds, indent=4)[:2000])

## Ghi chú

- File output `/kaggle/working/threshold.json` sẽ xuất hiện trong tab **Output** của Kaggle notebook → download về thay cho `threshold.json` gốc trong repo.
- Nếu bucket `gs://meta-cxr-checkpoint` là **private**: upload SA key JSON dưới dạng Kaggle Dataset, set `GOOGLE_APPLICATION_CREDENTIALS` ở cell 3.
- Mỗi encoder variant tune riêng: đổi `RUN_NAME` (`01_biovil_only`, `07_all_three`, …) và chạy lại.